In [1]:
import torch, os
os.environ["CUDA_VISIBLE_DEVICES"] = "2" 

In [2]:
import pandas as pd
import re
import os

# ==========================================
# 1. 設定路徑
# ==========================================
input_file = 'discharge.csv.gz'  # 來源檔案
output_file = 'processed_discharge.csv.gz'                 # 處理後輸出的新檔案

chunk_size = 5000  # 每次處理 5000 筆，避免記憶體不足

# ==========================================
# 2. 定義核心處理函數
# ==========================================
def truncate_before_physical_exam(text):
    """
    邏輯：尋找 'Physical Exam' 相關的標題，只保留該標題之前的文字。
    如果找不到該標題，則保留原始全文 (或是你可以選擇回傳空字串)。
    """
    if not isinstance(text, str):
        return ""
    
    # Regex 解釋：
    # (?i) : 忽略大小寫
    # \n\s*: 標題前必須有換行 (避免抓到句子中間提到的 "physical exam")
    # (?:Pertinent\s*)? : 可能會有 "Pertinent" 這個字
    # Physical\s+Exam : 核心關鍵字
    # (?:ination)? : 可能是 Exam 或是 Examination
    # \s*: : 結尾可能有冒號
    pattern = r'(?i)\n\s*(?:Pertinent\s*)?Physical\s+Exam(?:ination)?\s*:?'
    
    match = re.search(pattern, text)
    
    if match:
        # match.start() 是找到的標題起始位置
        # 我們只取從頭到這個位置的文字，並去除頭尾空白
        return text[:match.start()].strip()
    else:
        # 如果找不到 Physical Exam，這是一個風險點
        # 策略：保留原樣，但你可以考慮回傳 None 之後過濾掉
        return text 

# ==========================================
# 3. 分塊處理並寫入
# ==========================================
print(f"開始處理: {input_file} -> {output_file}")

# 如果輸出檔案已存在，先刪除避免重複寫入
if os.path.exists(output_file):
    os.remove(output_file)

processed_count = 0
cut_success_count = 0 # 統計有多少筆成功執行了切割

with pd.read_csv(input_file, compression='gzip', chunksize=chunk_size, on_bad_lines='skip') as reader:
    for i, chunk_df in enumerate(reader):
        
        # 備份原始長度用來比較 (非必要，除錯用)
        # original_lengths = chunk_df['text'].astype(str).apply(len)
        
        # --- 核心操作：直接覆蓋 text 欄位 ---
        # 1. 確保轉為字串
        chunk_df['text'] = chunk_df['text'].astype(str)
        
        # 2. 應用切割函數
        # 為了統計成功率，我們稍微分開寫
        original_texts = chunk_df['text'].tolist()
        new_texts = [truncate_before_physical_exam(t) for t in original_texts]
        
        # 3. 統計有多少筆真的變短了 (代表成功切除了後面的東西)
        for original, new in zip(original_texts, new_texts):
            if len(new) < len(original):
                cut_success_count += 1
        
        # 4. 覆寫回 DataFrame
        chunk_df['text'] = new_texts
        
        # --- 寫入檔案 ---
        # 第一個 chunk 寫入 header，之後的 append 不寫 header
        write_header = (i == 0)
        chunk_df.to_csv(output_file, mode='a', compression='gzip', index=False, header=write_header)
        
        processed_count += len(chunk_df)
        print(f"已處理: {processed_count} 筆 | 成功截斷比例: {cut_success_count/processed_count:.1%}", end='\r')

print(f"\n\n處理完成！檔案已儲存至: {output_file}")
print(f"總筆數: {processed_count}")
print(f"成功找到 'Physical Exam' 並截斷的筆數: {cut_success_count}")

# ==========================================
# 4. 檢查結果 (讀前 5 筆來看看)
# ==========================================
print("\n=== 檢查前 5 筆結果 ===")
df_check = pd.read_csv(output_file, compression='gzip', nrows=5)
pd.set_option('display.max_colwidth', 200) # 設定顯示寬度
print(df_check[['hadm_id', 'text']])

開始處理: discharge.csv.gz -> processed_discharge.csv.gz
已處理: 331793 筆 | 成功截斷比例: 96.1%

處理完成！檔案已儲存至: processed_discharge.csv.gz
總筆數: 331793
成功找到 'Physical Exam' 並截斷的筆數: 318714

=== 檢查前 5 筆結果 ===
    hadm_id  \
0  22595853   
1  22841357   
2  29079034   
3  25742920   
4  23052089   

                                                                                                                                                                                                      text  
0  Name:  ___                     Unit No:   ___\n \nAdmission Date:  ___              Discharge Date:   ___\n \nDate of Birth:  ___             Sex:   F\n \nService: MEDICINE\n \nAllergies: \nNo Kno...  
1  Name:  ___                     Unit No:   ___\n \nAdmission Date:  ___              Discharge Date:   ___\n \nDate of Birth:  ___             Sex:   F\n \nService: MEDICINE\n \nAllergies: \nPercoc...  
2  Name:  ___                     Unit No:   ___\n \nAdmission Date:  ___              Discharge Date: 

In [3]:
df_check

,note_id,subject_id,hadm_id,note_type,note_seq,charttime,storetime,text
0,10000032-DS-21,10000032,22595853,DS,21,2180-05-07 00:00:00,2180-05-09 15:26:00,Name: ___ Unit No: ___\n \nAdmission Date: ___ Discharge Date: ___\n \nDate of Birth: ___ Sex: F\n \nService: MEDICINE\n \nAllergies: \nNo Kno...
1,10000032-DS-22,10000032,22841357,DS,22,2180-06-27 00:00:00,2180-07-01 10:15:00,Name: ___ Unit No: ___\n \nAdmission Date: ___ Discharge Date: ___\n \nDate of Birth: ___ Sex: F\n \nService: MEDICINE\n \nAllergies: \nPercoc...
2,10000032-DS-23,10000032,29079034,DS,23,2180-07-25 00:00:00,2180-07-25 21:42:00,Name: ___ Unit No: ___\n \nAdmission Date: ___ Discharge Date: ___\n \nDate of Birth: ___ Sex: F\n \nService: MEDICINE\n \nAllergies: \nPercoc...
3,10000032-DS-24,10000032,25742920,DS,24,2180-08-07 00:00:00,2180-08-10 05:43:00,Name: ___ Unit No: ___\n \nAdmission Date: ___ Discharge Date: ___\n \nDate of Birth: ___ Sex: F\n \nService: MEDICINE\n \nAllergies: \nPercoc...
4,10000084-DS-17,10000084,23052089,DS,17,2160-11-25 00:00:00,2160-11-25 15:09:00,Name: ___ Unit No: ___\n \nAdmission Date: ___ Discharge Date: ___\n \nDate of Birth: ___ Sex: M\n \nService: MEDICINE\n \nAllergies: \nNo Know...


In [4]:
import pandas as pd

# 設定檔案路徑 (請確認這跟上一步驟輸出的檔名一致)
file_path = 'processed_discharge.csv.gz'

print(f"正在讀取 {file_path} 的前 5 筆資料...\n")

# 只讀取前 5 筆
df_check = pd.read_csv(file_path, compression='gzip', nrows=5)

# 確保是字串格式
df_check['text'] = df_check['text'].astype(str)

# 使用迴圈逐筆印出完整內容
for index, row in df_check.iterrows():
    print("=" * 60)
    print(f"Record ID: {index + 1} | HADM_ID: {row.get('hadm_id', 'N/A')}")
    print("=" * 60)
    
    # 直接印出完整字串，不會被縮減
    print(row['text'])
    
    print("\n" + "-" * 20 + " (End of this note) " + "-" * 20 + "\n\n")

正在讀取 processed_discharge.csv.gz 的前 5 筆資料...

Record ID: 1 | HADM_ID: 22595853
Name:  ___                     Unit No:   ___
 
Admission Date:  ___              Discharge Date:   ___
 
Date of Birth:  ___             Sex:   F
 
Service: MEDICINE
 
Allergies: 
No Known Allergies / Adverse Drug Reactions
 
Attending: ___
 
Chief Complaint:
Worsening ABD distension and pain 
 
Major Surgical or Invasive Procedure:
Paracentesis

 
History of Present Illness:
___ HCV cirrhosis c/b ascites, hiv on ART, h/o IVDU, COPD, 
bioplar, PTSD, presented from OSH ED with worsening abd 
distension over past week.  
Pt reports self-discontinuing lasix and spirnolactone ___ weeks 
ago, because she feels like "they don't do anything" and that 
she "doesn't want to put more chemicals in her." She does not 
follow Na-restricted diets. In the past week, she notes that she 
has been having worsening abd distension and discomfort. She 
denies ___ edema, or SOB, or orthopnea. She denies f/c/n/v, d/c, 
dysuria. Sh